# Characterize the small vessel skeleton generated from the binary small vessel (CD105) mask

The small vessel channel (anti-CD105 staining with Alexa546) was segmented with Imaris and exported as binary images.
The vessel skeleton was prepared using the "Skeletonize2d/3D plugin from Fiji, and the skeleton was analyzed using the "AnalyzeSkeleton" plugin.

Input: One of the two csv-files exported from the "AnalyseSkeleton" plugin in Fiji: The list with the detailed information is required.

Output: Average properties of the branches as a list in a txt-file, e.g. average branch length

In [1]:
# 1. Import all required libraries
import numpy as np
import matplotlib.pyplot as plt
import glob
import os

In [2]:
# 2. Define functions to calculate properties and export histograms
def get_properties(data: np.ndarray):
    avg_value = np.mean(data)
    min_value= np.min(data)
    max_value = np.max(data)
    stdev_value = np.std(data)
    median_value = np.median(data)
    
    return min_value, max_value, median_value, avg_value, stdev_value

def get_nanproperties(data: np.ndarray):
    avg_value = np.nanmean(data)
    min_value= np.nanmin(data)
    max_value = np.nanmax(data)
    stdev_value = np.nanstd(data)
    median_value = np.nanmedian(data)
    
    return min_value, max_value, median_value, avg_value, stdev_value

def get_histogram(data: np.ndarray, min_value: float = 100, max_value: float = 3000, nr_bins: int = 100):
    hist = plt.hist(data.flatten(), bins=nr_bins, range=(min_value,max_value))
    bins_edges = hist[1]
    counts = hist[0]
    bin_step = hist[1][1] - hist[1][0]
    last_bin = (min_value+bin_step/2) + nr_bins*bin_step
    bin_centers = np.arange(min_value+bin_step/2, last_bin, bin_step)
    
    return counts, bin_centers
    

In [3]:
# 3. folder to be processed
path = 'VK*.csv'
save_file_as = 'Properties_Skeleton_overlapping.txt'
save_filtered_as = 'Properties_Skeleton_above25um_overlapping.txt'

list_filenames = list()
list_properties = list()
list_nr_objects = list()
list_nr_large_objects = list()
list_properties_long = list()

In [4]:
# 4. loop over the folder to process all images
for file in glob.glob(path):
    # Get data
    filename = os.path.abspath(file).split(".")[0]
    data = np.genfromtxt(file ,skip_header=1, delimiter=',', usecols=(1, 8, 9))
    # Column 1: Branch length [um], Column 8: Euclidean length [um]
    # Column 9: Calibrated edge length calcualted with running average over 5 pixel
    
    # Calculate turtuosity
    tortuosity_column = np.where(data[:, 1] > 0, data[:, 0]/data[:, 1], np.nan)
    
    # Determine number of objects
    nr_objects = len(data[:,0])
    
    # Sort data
    for_sorting = np.array([data[:, 0], data[:, 1], data[:, 2], tortuosity_column]) 
    long_branches = np.where(for_sorting[2,:] > 25, for_sorting, np.nan)
    
    nan_objects = np.count_nonzero(np.isnan(long_branches[2,:]))
    nr_large_objects = nr_objects - nan_objects
    
    # Calculate mean and standard deviation for the parameter of interest
    branch_length = get_properties(data[:, 0])
    euclidean_length = get_properties(data[:, 1])
    running_average_length = get_properties(data[:, 2])
    tortuosity =get_nanproperties(tortuosity_column)
    
    props = np.concatenate([branch_length, euclidean_length, running_average_length, tortuosity])
    
    branch_length_long = get_nanproperties(long_branches[0, :])
    euclidean_length_long = get_nanproperties(long_branches[1, :])
    running_average_length_long = get_nanproperties(long_branches[2, :])
    tortuosity_long = get_nanproperties(long_branches[3, :])
    
    props_long = np.concatenate([branch_length_long, euclidean_length_long, running_average_length_long, tortuosity_long])
    
    # Determine total vessel length of all long vessel (> 20 µm)
    sum_branch_length = np.nansum(long_branches[0, :])
    print(filename, sum_branch_length)
    
    # Append all parameter to a growing list
    list_filenames.append(str(filename))
    list_nr_objects.append(nr_objects)
    list_properties.append(props)
    
    list_nr_large_objects.append(nr_large_objects)
    list_properties_long.append(props_long)

    header = 'Filename\t# Objects\tBranch Length min\tmax\tmedian\tmean\tstdev\tEuclidean Length min\tmax\tmedian\tmean\tstdev\tRunning Average Length min\tmax\tmedian\tmean\tstdev\tTortuosity min\tmax\tmedian\tmean\tstdev'

    # save results as txt file
    np.savetxt(
        save_file_as,
        np.vstack(
            [
                list_filenames,
                list_nr_objects,
                np.array(list_properties).T
            ],
        ).T,
        delimiter='\t', fmt="%s", header=header
    )
    
    np.savetxt(
        save_filtered_as,
        np.vstack(
            [
                list_filenames,
                list_nr_large_objects,
                np.array(list_properties_long).T
            ],
        ).T,
        delimiter='\t', fmt="%s", header=header
    )
    
    # Save txt file with new turtuosity column
    np.savetxt(
        filename + '_turt.txt',
        np.vstack(
            [
                data.T,
                tortuosity_column
            ],
        ).T,
        delimiter='\t', fmt="%s", header = 'Branch length [um]\tEuclidean length [um]\Running Average Length [um]\tTurtuosity'
    )

/tmp/ipykernel_117943/821958062.py:10: RuntimeWarning: divide by zero encountered in divide
  tortuosity_column = np.where(data[:, 1] > 0, data[:, 0]/data[:, 1], np.nan)


/srv/cui-bee/users/AG Stegner/Femur-Gefaessbaum/13_Shaft_automatic_sorting/OverlappingRegions_SmallVessel/CD105/Skeleton/Skeleton_Analyse/VK-AA498_shaft_1018-1697_skeleton 682364.249
/srv/cui-bee/users/AG Stegner/Femur-Gefaessbaum/13_Shaft_automatic_sorting/OverlappingRegions_SmallVessel/CD105/Skeleton/Skeleton_Analyse/VK-AA498_shaft_1697-2376_skeleton 802695.8080000001
/srv/cui-bee/users/AG Stegner/Femur-Gefaessbaum/13_Shaft_automatic_sorting/OverlappingRegions_SmallVessel/CD105/Skeleton/Skeleton_Analyse/VK-AA498_shaft_2376-3055_skeleton 775728.443
/srv/cui-bee/users/AG Stegner/Femur-Gefaessbaum/13_Shaft_automatic_sorting/OverlappingRegions_SmallVessel/CD105/Skeleton/Skeleton_Analyse/VK-AA498_shaft_339-1018_skeleton 512283.79699999996
/srv/cui-bee/users/AG Stegner/Femur-Gefaessbaum/13_Shaft_automatic_sorting/OverlappingRegions_SmallVessel/CD105/Skeleton/Skeleton_Analyse/VK-AA499_shaft_1475-2065_skeleton 642377.432
/srv/cui-bee/users/AG Stegner/Femur-Gefaessbaum/13_Shaft_automatic_sort